# Downsampling Eaglei Data and Merging with County and ERA5 Data

The goal of this notebook is to:
- Downsample the eaglei data for each year to a 6-hour cadence, replacing the number of customers without power with the 6-hour mean
- Combine data for all years into a single file
- Export the combined data
- Merge additional county info
- Export the merged data

What I don't have working yet:
- Look up info from the ERA5 dataset about temperature, precipitation, snow depth, and wind speed

(I'm getting stuck on merging the weatherbench xarray with the eaglei data)

In [1]:
import pandas as pd
import dask.dataframe as dd
import numpy as np
import os

### Helper functions for downsampling

The eaglei data only includes entries when the customers_out value is nonzero.

In order to fill in all the missing values, we'll need to add additional (zero) entries to the eaglei data for the first and last possible timestamp of each year

The functions below are used to assist in this process

## Load and Merge the eaglei data

We'll merge all the eaglei_outages files into a single dataframe and remove entries that correspond to anything other than the lower 48 states

In [2]:
#Get a list of files that start with eaglei_outages_ in the directory ../Data/eaglei_data/
filelist = os.listdir('../Data/eaglei_data/')
files = [f for f in filelist if f.startswith('eaglei_outages_2')]

# Use dask to open all the files in the files list and merge them into a single dataframe
df = dd.read_csv(['../Data/eaglei_data/' + f for f in files], assume_missing=True, dtype={'fips_code': 'object'})

# Convert the dask dataframe to a pandas dataframe
df = df.compute()

# Drop the county and state variables
df = df.drop(columns=['county', 'state'])

# Convert the run_start_time column to datetime
df['run_start_time'] = pd.to_datetime(df['run_start_time'])

# Convert the customers_out column to numeric
df['customers_out'] = pd.to_numeric(df['customers_out'], errors='coerce')

# Drop all rows with fips_code values that correspond to Alaska, Hawaii, or non-states
df = df[~df['fips_code'].str.startswith(('02', '03', '07', '14', '15', '43', '52'))]
df = df[~df['fips_code'].str.startswith(('6', '7', '8', '9'))]

In [ ]:
#Export df to a parquet file
df.to_parquet('../Data/eaglei_data/eaglei_outages_all_unfilled.parquet', index=False)

In [3]:
# Load df from the parquet file
df = pd.read_parquet('../Data/eaglei_data/eaglei_outages_all_unfilled.parquet')

## Downsample the eaglei data

The data appear to only exist when the number of customers without power is nonzero. So we'll first need to upsample to a true 15-minute cadence and fill in missing values with 0.
(Note that this is a bit of an assumption that only non-zero values are being reported, rather than values being missing.)

Then we'll downsample to a 6-hour cadence by computing the mean number of customers out over the 6-hour window.

Note that this code blows through the available memory on my computer when I try to do the merge in pandas. So I'll be generating the multiindex in pandas, export it as a file, and then load both the multiindex and available data and merge them using dask before then doing the filling. So there will be several saving/loading steps.

### Start by creating an empty dataframe with all combinations of fips_code and time

In [ ]:
# Create datetimes for the first and last timestamps of the year
start = pd.to_datetime('2014-11-01 00:00:00')
end = pd.to_datetime('2023-12-31 23:45:00')

# Create a date range from start to end with 15 minute intervals
date_range = pd.date_range(start, end, freq='15min')

# Create a DataFrame with all combinations of fips_code and the date range
fips_codes = df['fips_code'].unique()

# Create a dask dataframe all_combinations with the index from the product of fips_codes and date_range with columns 'fips_code' and 'run_start_time'
all_combinations = pd.MultiIndex.from_product([fips_codes, date_range], names=['fips_code', 'run_start_time'])

all_df = pd.DataFrame(index=all_combinations).reset_index()

# Export all_df as a parquet file
all_df.to_parquet('../Data/eaglei_data/eaglei_outages_all_combinations.parquet', index=False)

date range created
fips codes listed


In [2]:
# Load the dataframe with the outage data
df = dd.read_parquet('../Data/eaglei_data/eaglei_outages_all_unfilled.parquet', dtype={'fips_code': 'object'})

# Create a new variable that concatenates fips_code and run_start_time
df['fips_run_start'] = df['fips_code'].astype(str) + '_' + df['run_start_time'].astype(str)

# Set fips_run_start as the index
df = df.set_index('fips_run_start')

# Load the dataframe with all fips-datetime combinations
df_all = dd.read_parquet('../Data/eaglei_data/eaglei_outages_all_combinations.parquet', dtype={'fips_code': 'object'})

# Create a new variable that concatenates fips_code and run_start_time
df_all['fips_run_start'] = df['fips_code'].astype(str) + '_' + df['run_start_time'].astype(str)

# Set fips_run_start as the index
df_all = df.set_index('fips_run_start')

# Use dask to merge df_all and df on their indices, using the index from df_all as the join key
df_all_merged = df_all.merge(df, how='left', left_index=True, right_index=True)

# Drop the columns fips_code_y, run_start_time_y, and customers_out_x
df_all_merged = df_all_merged.drop(columns=['fips_code_y', 'run_start_time_y', 'customers_out_x'])

# Rename the columns fips_code_x as fips_code, run_start_time_x as run_start_time, and customers_out_y as customers_out
df_all_merged = df_all_merged.rename(columns={'fips_code_x': 'fips_code', 'run_start_time_x': 'run_start_time', 'customers_out_y': 'customers_out'})

# Use dask to merge all_df with df on=['fips_code', 'run_start_time']
#all_df = dd.merge(all_df, df, on=['fips_code', 'run_start_time'], how='left')

# Fill missing values in customers_out with 0
df_all_merged['customers_out'] = df_all_merged['customers_out'].fillna(0)

#In the fips_code variable, replace 0 with NA
df_all_merged['fips_code'] = df_all_merged['fips_code'].replace(0, np.nan)

#Forward fill the fips_code with the most recent value
df_all_merged['fips_code'] = df_all_merged['fips_code'].ffill()

# Set run_start_time as the index
#all_df.set_index('run_start_time', inplace=True)

In [ ]:
# Export df_all_merged as a parquet file partitioning on fips_code
df_all_merged.to_parquet('../Data/eaglei_data/eaglei_outages_all_filled_upsampled.parquet', index=False, partition_cols=['fips_code'])

In [10]:
#Group the data by fips_code. Then downsample the data to every 6 hours, replacing customers_out with the mean and then ungroup the data
df_all_merged = df_all_merged.groupby('fips_code').set_index('run_start_time').resample('6h').mean().reset_index(level=0, drop=True)

AttributeError: 'Column not found: set_index'

In [ ]:
#For exporting and future merging, move the datetime back to a "regular" variable and reset the index
df['datetime'] = pd.to_datetime(outages.index)
df.reset_index(drop=True, inplace=True)

In [16]:
#Export outages to a parquet
outages.to_parquet('../Data/eaglei_data/eaglei_outages.parquet')

In [3]:
#Or, if the parquet file has already been saved, load the parquet
outages = pd.read_parquet('../Data/eaglei_data/eaglei_outages.parquet')

Next, we can merge the outages data with the additional county variables from the Counties_All file

In [20]:
#Add a YEAR variable from datetime
outages['YEAR'] = outages['datetime'].dt.year

#Load ../Data/Counties_All.csv
counties = pd.read_csv('../Data/County_level_Variables/Counties_All.csv')

# Merge outages and counties based on the YEAR variable and the fips_code/FIPS variables
outages_merged = outages.merge(counties, left_on=['YEAR', 'fips_code'], right_on=['YEAR', 'FIPS'])

In [21]:
#Export outages_merged to a parquet
outages_merged.to_parquet('../Data/Merged_Data/eaglei_outages_with_county_info.parquet')

In [ ]:
#Or, if the parquet file has already been saved, load the parquet
outages_merged = pd.read_parquet('../Data/Merged_Data/eaglei_outages_with_county_info.parquet')

## Checking for Missing Counties

It looks like some counties are missing data for some years. We'll check on this.

We'll:
- Make a dataframe (fips_codes_df) that lists the fips_code values present in each year's dataset and creates a set of all fips_code values from all years
- Make a dataframe (fips_codes_missing_df) that lists the fips_code values that don't have any data for each year
- Make a dataframe (missing_fips_df) that has one row per missing fips_code and indicates whether or not the code was missing in each year (1) or not (0) from 2014 to 2023

In [ ]:
#Get a list of files that start with eaglei_outages_ in the directory ../Data/eaglei_data/
files = os.listdir('../Data/eaglei_data/')
files = [f for f in files if f.startswith('eaglei_outages_2')]

#Create an empty set
fips_codes = set()

#Create an empty data frame with the variables fips and year
fips_codes_df = pd.DataFrame(columns=['year', 'fips'])

i=0

for filename in files:
    #Open the file
    df = pd.read_csv('../Data/eaglei_data/' + filename)

    #Extract a sequence of four digits from filename
    year_local = int(filename.split('_')[2].split('.')[0])

    #Create a set of fips_code values for the loaded file
    fips_codes_local = set(df['fips_code'].unique())

    #Add the new set of fips_codes_local to the fips_codes set
    fips_codes = fips_codes.union(fips_codes_local)

    # Add a row to fips_codes_df with year=year_local and fips=fips_codes_local
    fips_codes_df.loc[i] = [year_local, fips_codes_local]
    i=i+1

In [ ]:
fips_codes_missing_df = pd.DataFrame(columns=['year', 'fips_missing'])

i=0
for i in range(len(fips_codes_df)):
    fips_codes_local = fips_codes_df.loc[i]['fips']
    year = fips_codes_df.loc[i]['year']
    # Create a set of values that are in fips_codes but not in fips_codes_local
    missing_fips = fips_codes - fips_codes_local

    fips_codes_missing_df.loc[i] = [year, missing_fips]

    i=i+1

#In fips_codes_missing_df convert year to the index
fips_codes_missing_df.set_index('year', inplace=True)

In [ ]:
# Create a set that is the union of all sets in fips_codes_missing_df['fips_missing']
missing_fips = set()
for i in range(len(fips_codes_missing_df)):
    missing_fips = missing_fips.union(fips_codes_missing_df.loc[i]['fips_missing'])

# Create a pandas dataframe with columns fips, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, and 2023
missing_fips_df = pd.DataFrame(columns=['fips', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023'])

# For each item in missing_fips_years, create a list that consists of the key, a 1 for each year (in 2014-2023) that is in the values, and a 0 for each year that is not in the values
i=0
for fips in missing_fips:
    yearlist = [fips]
    # Create a list of 0s and 1s for each year from 2014 to 2023
    for year in range(10):
        currentyear = 2014+year
        tempset = fips_codes_missing_df.loc[currentyear]['fips_missing']
        if fips in tempset:
            yearlist.append(1)
        else:
            yearlist.append(0)
        # Add the fips code and the list of values to the dataframe
    missing_fips_df.loc[i] = yearlist
    i=i+1

# Sort missing_fips_df by fips
missing_fips_df.sort_values(by='fips', inplace=True)

In [100]:
# Export missing_fips_df to a csv
missing_fips_df.to_csv('../Data/Missing_FIPS.csv', index=False)

In [ ]:
# We can also create a dictionary that uses the fips_code as a key and its missing years as values:
missing_fips_years = {}
for fips in missing_fips:
    missing_fips_years[fips] = []
    for i in range(len(fips_codes_missing_df)):
        if fips in fips_codes_missing_df.loc[i]['fips_missing']:
            missing_fips_years[fips].append(fips_codes_missing_df.loc[i]['year'])
